# 3D IR-bSSFP radial "Kooshball" sequence
The `.seq` file is written to `out_dir`.

In [ ]:
import os
import time
import numpy as np
import torch
import pypulseq as pp
from typing import Optional, Tuple
from pypulseq import Sequence
from Modules.CoilSens import cyclic_head_coil
from Modules import system, load_brain_phantom

# ---- output folder for the sequence file ------------------------------------
out_dir  = r"./sequences"          # <-- The sequence will be written to this target folder
seq_name = "bssfp_kooshball3D_signal_n64_R384_S25.seq"   # <-- Change file name
os.makedirs(out_dir, exist_ok=True)

torch.set_default_device("cuda")
device = torch.device("cuda")

In [ ]:
device = torch.device("cuda")

# ---- single source of truth ------------------------------------------------
FOV        = 250e-3
n          = 64      # phantom + recon matrix (n^3)
N_x        = n        # readout samples / spoke  -> kmax matches the recon grid
N_readout  = 384      # echoes per inversion segment
N_segments = 25       # inversion segments (25*384 = 9600 spokes)
n_adc      = N_segments * N_readout * N_x   # total ADC samples (8 coils apart)
N_dummy    = 0
slice_thickness = 64e-3
TI_START   = 20e-3    # TI of the first echo; the train then spans ~20 ms -> ~3.1 s
TI_inter   = 4.88e-3  # recovery delay between echoes (echo spacing P = TR + TI_inter)
N_channels = 8
spin_count = 64       # isochromats per voxel (simulation only)

# Tissue ground truth (as defined in Modules/phantom.py):
#   label 1 -> CSF: PD 1.0, T1 4.0 s, T2 0.5 s
#   label 2 -> GM : PD 0.8, T1 1.3 s, T2 0.08 s
#   label 3 -> WM : PD 0.7, T1 0.9 s, T2 0.07 s
#   T2* = 0.04 s everywhere
TISSUES = {
    "WM":  dict(T1=0.9, T2=0.07, PD=0.7),
    "GM":  dict(T1=1.3, T2=0.08, PD=0.8),
    "CSF": dict(T1=4.0, T2=0.5,  PD=1.0),
}
T2STAR = 0.04
NAMES = ("WM", "GM", "CSF")

# ---- phantom (BrainWeb subject04) ------------------------------------------
phantom = load_brain_phantom(
    raw_path="subject04_crisp_v.rawb",
    target_size=(n, n, n),
    device=device,
    permute_order=(2, 1, 0),
)
print(type(phantom).__name__, "size =", phantom.size)

pd_map = phantom.PD.detach().cpu().numpy()

brain = pd_map > 0.10                       # exclude background + thin rim
masks = {
    "WM":  brain & (pd_map < 0.75),
    "GM":  (pd_map >= 0.75) & (pd_map < 0.90),
    "CSF": (pd_map >= 0.90),
}
brain_mask = brain
for t in NAMES:
    print(f"mask {t}: {int(masks[t].sum())} voxels")

# ---- coils (used directly - identity orientation, no auto-match) -----------
scale = tuple(float(s) / FOV for s in phantom.size)
coil_sim = cyclic_head_coil((N_channels, n, n, n), fov_scaling=scale).to(torch.complex64)
phantom.coil_sens = coil_sim.to(device)
phantom.B0 = 0 * phantom.B0
phantom.B1 = torch.ones(phantom.B1.shape, dtype=phantom.B1.dtype)
data = phantom.build()
print("simulation voxels:", int(data.PD.numel()))

# combine weights for the SENSE/adaptive combine of the NUFFT previews
coil_c64 = coil_sim.to(device)
coil_den = (coil_c64.abs() ** 2).sum(0)

## Build the bSSFP sequence (constant 180 deg RF cycling)
IR balanced-SSFP (alpha = 60 deg) 3D radial sequence. Each of the 25 segments:
inversion -> TI_start -> N_readout bSSFP echoes -> TD, golden-angle radial
readout per echo.

The RF/ADC phase offset advances by a CONSTANT `rf_spoiling_inc` (180 deg)
per echo, the steady-state phase cycle of bSSFP. With an even N_readout,
echo p of every segment has the same RF phase, so the per-echo-index frames
used for the T1* fit are mutually coherent.

In [ ]:
t1 = time.time()

# ---- sequence parameters (formerly function arguments) ----------------------
alpha_deg        = 60.0
TE               = 3.98e-3
TD               = 2000e-3
rf_spoiling_inc  = 180.0       # bSSFP steady-state cycle
plot_seq         = True

system = default_system()
seq = pp.Sequence(system)
deltak = 1 / FOV

rf_inv, gz_inv, _ = pp.make_sinc_pulse(
    flip_angle=np.pi, duration=10e-3, slice_thickness=slice_thickness,
    apodization=0.5, time_bw_product=4, system=system,
    return_gz=True, use='inversion')

rf, gz, _ = pp.make_sinc_pulse(
    apodization=0.5, duration=2e-3, flip_angle=alpha_deg*np.pi/180,
    slice_thickness=slice_thickness, system=system,
    time_bw_product=4, return_gz=True, use='excitation')

gx_base = pp.make_trapezoid(channel='x', flat_area=N_x*deltak, flat_time=6.4e-3/5, system=system)
adc = pp.make_adc(num_samples=N_x, duration=gx_base.flat_time, delay=gx_base.rise_time, system=system)
gx_pre_base = pp.make_trapezoid(channel='x', area=-gx_base.area/2, duration=2e-3, system=system)
gz_reph = pp.make_trapezoid(channel='z', area=-gz.area/2, duration=2e-3, system=system)

# z gradient played right after the inversion pulse. It cancels the
# slab-select area of the inversion (gz_inv.area) and leaves -gz.area/2,
# i.e. exactly the pre-compensation the first excitation needs.
gz_post_inv = pp.make_trapezoid(
    channel='z', area=-gz_inv.area - gz.area/2, system=system)

delay_TE = (
    np.ceil((
        TE - pp.calc_duration(gx_pre_base)
        - gz.fall_time - gz.flat_time/2
        - pp.calc_duration(gx_base)/2
    ) / seq.grad_raster_time) * seq.grad_raster_time
)
inv_duration   = pp.calc_duration(rf_inv, gz_inv)
delay_TI_start = (
    np.ceil((TI_START - inv_duration/2 - pp.calc_duration(gz)/2)
            / seq.grad_raster_time) * seq.grad_raster_time
)
assert delay_TI_start > 0, 'TI_start too short'
assert pp.calc_duration(gz_post_inv) <= delay_TI_start, 'post-inversion z gradient does not fit in TI_start'
delay_TI_inter = np.ceil(TI_inter / seq.grad_raster_time) * seq.grad_raster_time

gre_block_dur = (
    pp.calc_duration(gz) + pp.calc_duration(gx_pre_base)
    + delay_TE + pp.calc_duration(gx_base)
    + pp.calc_duration(gx_pre_base)
)
delay_TD = np.ceil(TD / seq.grad_raster_time) * seq.grad_raster_time

total_acquired = N_segments * N_readout
rf_phase = 0.0
spoke_counter = 0

for seg in range(N_segments):
    seq.add_block(rf_inv, gz_inv)
    seq.add_block(pp.make_delay(delay_TI_start), gz_post_inv)   # z balance after inversion

    n_this_seg = N_readout + (N_dummy if seg == 0 else 0)
    for s in range(n_this_seg):
        is_dummy = (seg == 0) and (s < N_dummy)
        is_last  = (s == n_this_seg - 1)

        rf.phase_offset  = rf_phase / 180 * np.pi
        adc.phase_offset = rf_phase / 180 * np.pi
        # CONSTANT bSSFP phase cycle (180 deg/echo)
        rf_phase = (rf_phase + rf_spoiling_inc) % 360

        if is_dummy:
            kx, ky, kz = 1.0, 0.0, 0.0
        else:
            # golden-angle 3D radial spoke direction (inlined)
            phi   = np.arccos(1 - 2*(spoke_counter + 0.5)/total_acquired)
            theta = np.pi*(1 + np.sqrt(5))*spoke_counter
            kx, ky, kz = (np.sin(phi)*np.cos(theta), np.sin(phi)*np.sin(theta), np.cos(phi))
            spoke_counter += 1

        seq.add_block(rf, gz)
        seq.add_block(
            pp.make_trapezoid(channel='x', area=gx_pre_base.area*kx, duration=2e-3, system=system),
            pp.make_trapezoid(channel='y', area=gx_pre_base.area*ky, duration=2e-3, system=system),
            pp.make_trapezoid(channel='z', area=gx_pre_base.area*kz+gz_reph.area, duration=2e-3, system=system),
        )
        seq.add_block(pp.make_delay(delay_TE))

        gx = pp.make_trapezoid(channel='x', amplitude=gx_base.amplitude*kx,
                    flat_time=gx_base.flat_time, rise_time=gx_base.rise_time, fall_time=gx_base.fall_time, system=system)
        gy = pp.make_trapezoid(channel='y', amplitude=gx_base.amplitude*ky,
                    flat_time=gx_base.flat_time, rise_time=gx_base.rise_time, fall_time=gx_base.fall_time, system=system)
        gz_ro = pp.make_trapezoid(channel='z', amplitude=gx_base.amplitude*kz,
                    flat_time=gx_base.flat_time, rise_time=gx_base.rise_time, fall_time=gx_base.fall_time, system=system)

        if is_dummy: seq.add_block(gx, gy, gz_ro)
        else: seq.add_block(gx, gy, gz_ro, adc)

        # rewinder: x/y undo the prephaser; z undoes the kz blip and
        # pre-compensates the next slab-select lobe (-gz.area/2), except in
        # the last TR of a segment, where the z moment must end at 0.
        seq.add_block(
            pp.make_trapezoid(channel='x', area=gx_pre_base.area*kx, duration=2e-3, system=system),
            pp.make_trapezoid(channel='y', area=gx_pre_base.area*ky, duration=2e-3, system=system),
            pp.make_trapezoid(channel='z', area=gx_pre_base.area*kz + (0.0 if is_last else gz_reph.area),
                              duration=2e-3, system=system),
        )
        if s < n_this_seg - 1:
            seq.add_block(pp.make_delay(delay_TI_inter))

    seq.add_block(pp.make_delay(delay_TD))

ok, err = seq.check_timing()
if ok:
    print('bSSFP timing check passed')
    print(f'  GRE block dur : {gre_block_dur*1000:.1f} ms')
    print(f'  Echo spacing P: {(gre_block_dur+delay_TI_inter)*1000:.1f} ms')
    print(f'  Readout train : {N_readout*(gre_block_dur+delay_TI_inter)*1000:.0f} ms')
    est = N_segments*(inv_duration+delay_TI_start+N_readout*gre_block_dur
                      +(N_readout-1)*delay_TI_inter+TD)
    print(f'  Scan time est : {est/60:.1f} min')
else:
    print(f'Timing FAILED: {err}')

seq.set_definition('FOV', [FOV, FOV, slice_thickness])
seq.set_definition('Name', seq_name)

# ---- write the sequence to the chosen folder --------------------------------
os.makedirs(out_dir, exist_ok=True)
seq_path = os.path.join(out_dir, seq_name)
seq.write(seq_path)
print("sequence written to:", os.path.abspath(seq_path))

if plot_seq:
    seq.plot(time_range=(0.2, 0.4))

# values the downstream cells expect (previously the function's return values)
TR_echo = gre_block_dur

In [ ]:
P_echo = TR_echo + delay_TI_inter
alpha = 60.0 * np.pi / 180
print(f"build: {time.time()-t1:.0f} s   P = {P_echo*1e3:.2f} ms")

# effective TI of echo index p (p = 0..N_readout-1): the train starts at
# TI_START and each echo index advances by one echo spacing P.
ti_effs = np.array([TI_START + p * P_echo for p in range(N_readout)])

def null_ti(T1, P, TR=TR_echo, TD=TD, alpha=alpha, N=N_readout):
    # TI at which the first excitation of each (cyclic) segment sees Mz = 0.
    # Cyclic state: the magnetization just before the inversion pulse is the
    # fixed point of one segment (inversion -> TI -> N echoes -> TD).
    E1 = np.exp(-P / T1)
    lam = E1 * np.cos(alpha)
    kap = (1 - E1) / (1 - E1 * np.cos(alpha))          # Mz_ss / M0
    E_tot = np.exp(-(TR + TD) / T1)
    z_star = (1 - E_tot) + E_tot * np.cos(alpha) * kap * (1 - lam ** (N - 1))
    return T1 * np.log(1.0 + z_star)

TI_null = {t: null_ti(p["T1"], P_echo) for t, p in TISSUES.items()}
null_frame = {t: int(np.argmin(np.abs(ti_effs - TI_null[t]))) for t in TI_null}
for t in NAMES:
    print(f"TI_null({t}) = {TI_null[t]*1e3:7.1f} ms   frame {null_frame[t]:3d} "
          f"(TI_eff = {ti_effs[null_frame[t]]*1e3:5.0f} ms)")